# RNN/LSTM 与序列状态

## 学习目标

能够解释 batch、time、feature 维度以及 LSTM 隐藏状态和单元状态。


## 概念模型与执行路径

RNN 在每个时间步复用参数并传递状态。LSTM 使用门控缓解长期依赖中的梯度问题。`batch_first=True` 只改变输入输出布局，不改变隐藏状态布局。


### 实验 1：定位序列模型组件

**实验目的**：定位课程根目录，为导入合成序列数据与共享 LSTM 分类器做准备。路径查找失败通常意味着 Jupyter 启动目录不在预期位置，而不是 RNN 本身出错。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：检查多层 LSTM 的输入、输出与状态

**实验目的**：明确 batch、time、feature 三个维度以及两层 LSTM 的状态布局。`batch_first=True` 让输入和逐时间步输出使用 `(batch,time,feature)`，所以输入是 `(4,7,3)`，`outputs` 是 `(4,7,5)`。

`hidden` 与 `cell` 的 shape 都是 `(num_layers*num_directions,batch,hidden_size)=(2,4,5)`，不受 `batch_first` 影响。`hidden[0]` 是第一层最终隐藏状态，`hidden[1]` 是第二层最终隐藏状态；cell 保存 LSTM 的长期记忆通道。未显式传初始状态时，PyTorch 使用全零状态。

**机制**：第一层为每个时间步产生 5 维输出并作为第二层输入；`outputs` 只包含顶层在所有时间步的隐藏状态，而 `hidden`/`cell` 保存每一层最后时刻的状态。

In [ ]:
import torch
from torch import nn
lstm = nn.LSTM(input_size=3, hidden_size=5, num_layers=2, batch_first=True)
sequence = torch.randn(4, 7, 3)
outputs, (hidden, cell) = lstm(sequence)
print("outputs:", outputs.shape)
print("hidden:", hidden.shape, "cell:", cell.shape)


### 实验 3：验证顶层最后输出与最终隐藏状态

**实验目的**：在单向、无 padding 的 LSTM 中验证 `outputs[:, -1] == hidden[-1]`。左侧取顶层输出序列的最后时间步，右侧取最后一层的最终隐藏状态，shape 都是 `(4,5)`。

这个等式有边界：双向 LSTM 的反向最终状态不对应 `outputs[:, -1]` 的同一切片；含 padding 的 batch 中 `-1` 可能是补齐位置，应使用真实长度、packing 或按长度索引。不要机械地把最后数组位置当成每条序列的最后有效状态。

In [ ]:
torch.testing.assert_close(outputs[:, -1], hidden[-1])
print("last output equals top-layer final hidden state")


### 实验 4：理解递增/递减合成序列数据

**实验目的**：检查终端训练示例使用的单个样本。`make_dataset(size=8,steps=12)` 返回 8 条 `(12,1)` 序列及二分类标签。label 1 对应斜率 +1，label 0 对应斜率 -1。

每条序列由随机起点、线性趋势和标准差 0.03 的噪声组成。时间轴是 `0/12 ... 11/12`，所以理想首尾差约为 `±11/12`，而不是精确 ±1；噪声还会产生小偏差。分类需要识别随时间的方向，而不是依赖绝对起点。

**随机性**：本单元没有先固定种子，单独运行时 label 和数值会变化，但 shape 与标签语义保持不变。

In [ ]:
from examples.rnn_sequences import make_dataset
dataset = make_dataset(size=8, steps=12)
sample, label = dataset[0]
print("sample shape:", sample.shape, "direction label:", label.item())
print("first/last value:", sample[0].item(), sample[-1].item())


### 实验 5：训练序列方向分类器

**实验目的**：运行完整的 LSTM 分类任务。命令需要在终端执行；quick 模式生成 128 条序列，80% 训练、20% 验证，训练 5 个 epoch。

共享 `SequenceClassifier` 使用单层 `LSTM(input_size=1,hidden_size=32,batch_first=True)`，取 `hidden[-1]` 经 `Linear(32,2)` 输出 logits。训练使用 Adam、交叉熵和 shuffle，验证关闭梯度。该数据任务较简单，验证准确率应随训练提高，但小验证集会有离散波动。

**边界**：脚本固定 `seed_everything()`，但 `random_split` 依赖随后全局 RNG 状态；在相同软件环境和执行路径下可复现。它没有保存 checkpoint，也没有处理变长序列。

In [ ]:
# python 07-deep-learning/pytorch/examples/rnn_sequences.py --quick --epochs 5


## 底层机制

RNN 在时间维复用同一组参数，因此序列越长，计算图沿时间展开得越深。LSTM 通过 input、forget、output gate 和 cell state 为信息与梯度提供受控路径，缓解但不能彻底消除长程依赖问题。

隐藏状态 shape 始终是 `(layers*directions,batch,hidden)`。变长序列可用 padding 加 mask，或 `pack_padded_sequence` 跳过补齐计算；若跨 batch 延续状态，必须保证样本连续且在截断反向传播边界对状态 `detach()`，否则计算图跨 batch 增长。

## 官方教程补充

**对应官方源文件：** `beginner_source/nlp/sequence_models_tutorial.py`、`intermediate_source/char_rnn_classification_tutorial.py`、`intermediate_source/char_rnn_generation_tutorial.py`

官方序列教程强调区分 batch、time、feature 三个维度，以及输出序列与最终 hidden state。LSTM 同时维护 hidden/cell state；跨 batch 传递状态时若不想反向穿过整段历史，需要 detach。变长序列要用 padding mask、pack 或长度感知的聚合，不能把 padding 当真实 token。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

回答并验证：1）删除 `batch_first=True` 后输入和 outputs 如何排列，hidden 是否变化？2）`outputs` 为什么只含顶层状态？3）何时 `outputs[:,-1]` 不等于想要的最终有效状态？4）双向 LSTM 的 hidden 第一维如何编号？5）跨 batch 保留状态为何需要 detach？6）当前数据的 label 0/1 分别代表什么？

## 试一试

把 LSTM 改成双向，先预测 outputs 与 hidden shape，再把分类头改为接收两个方向最终状态的拼接。生成不同长度序列，用 padding 与 packing 分别训练并比较。最后跨多个短片段传递状态，实现截断反向传播并监控图和内存是否增长。

## 常见错误与调试

- **混淆 batch/time**：模型能运行却把样本当时间步；在边界断言 shape。
- **分类时取错层/方向**：`hidden[0]` 不一定是顶层；按 layers×directions 解读。
- **把 padding 当真实时间步**：最后状态被补齐污染；使用长度索引或 packing。
- **跨 batch 保留状态不 detach**：图不断增长并重复反传；在截断边界分离状态。
- **状态 batch size 不匹配**：最后 batch 更小时旧状态无法复用；重建或切片。
- **只看最后值解决趋势任务**：可能学到捷径；随机化起点并分析分布。